In [ ]:
# %% [markdown]
# # 🏥 Patient No-Show Prediction - Datathon Final Notebook
# 
# Bu notebook, bir Datathon kapsamında geliştirilen, hastaların sağlık randevularına gelmeme (no-show) durumlarını tahmin eden uçtan uca bir makine öğrenmesi boru hattını (pipeline) içermektedir.
# 
# ### 🎯 Projenin Öne Çıkan Özellikleri:
# * **Gelişmiş Özellik Mühendisliği:** Zaman farklarından yeni metrikler ve hedef tabanlı kodlama (Target Encoding).
# * **Sınıf Dengesizliği Yönetimi:** Ölçeklendirilmiş pozitif sınıf ağırlıkları (Scale Pos Weight) ile F1 ve PR-AUC optimizasyonu.
# * **Ensemble (Topluluk) Öğrenmesi:** XGBoost, LightGBM ve CatBoost modellerinin olasılık tabanlı ağırlıklandırılması.
# * **Seed Averaging:** Modellerin rastgale faktörlerden arındırılması için çoklu seed eğitimi.
# * **Pseudo Labeling (Yarı Gözetimli Öğrenme):** Test setindeki yüksek güvenli tahminlerin eğitim verisine dahil edilmesi.

# %%
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split

print("Kütüphaneler başarıyla yüklendi.")

# %% [markdown]
# ## 1. Veri Setlerinin Yüklenmesi
# Ham eğitim ve test verileri sisteme dahil ediliyor.

# %%
train = pd.read_csv('appointments_train.csv')
test  = pd.read_csv('appointments_test.csv')

print(f"Eğitim Verisi Boyutu: {train.shape}")
print(f"Test Verisi Boyutu: {test.shape}")

# %% [markdown]
# ## 2. Target Encoding İçin Referans Veri Seti Oluşturma
# Veri sızıntısını (Data Leakage) asgariye indirmek ve dönüşüm mantığını oturtmak için ham verinin bir kopyası referans (train_raw) olarak ayrılıyor.

# %%
train_raw = train.copy()
train_raw["appointment_datetime"] = pd.to_datetime(train_raw['appointment_datetime'])
train_raw["booking_datetime"]     = pd.to_datetime(train_raw['booking_datetime'])
train_raw["spec_type_key"]        = train_raw['specialty'].astype(str) + '_' + train_raw['appointment_type'].astype(str)
train_raw["appointment_datetime_month"] = train_raw['appointment_datetime'].dt.month
train_raw["lead_time_day"]        = (train_raw['appointment_datetime'] - train_raw['booking_datetime']).dt.days

# %% [markdown]
# ## 3. Gelişmiş Özellik Mühendisliği (Feature Engineering)
# Zaman damgalarından bekleme süreleri, saat dilimleri, hafta sonu etkileri türetiliyor ve kategorik değişkenler için Target Encoding uygulanıyor.

# %%
def add_features(df, ref):
    df = df.copy()

    # Datetime Dönüşümleri
    df["appointment_datetime"] = pd.to_datetime(df['appointment_datetime'])
    df["booking_datetime"]     = pd.to_datetime(df['booking_datetime'])

    # Bekleme Süresi Metrikleri (Lead Time)
    df["lead_time_day"]   = (df['appointment_datetime'] - df['booking_datetime']).dt.days
    df["lead_time_hours"] = (df['appointment_datetime'] - df['booking_datetime']).dt.total_seconds() / 3600
    df["lead_time_bucket"] = pd.cut(
        df['lead_time_day'],
        bins=[-1, 0, 1, 3, 7, 14, 30, 999],
        labels=[0,1,2,3,4,5,6]
    ).astype(int)

    # Kategorik Durum Belirteçleri
    df['is_same_day'] = (df['lead_time_day'] == 0).astype(int)
    df['is_weekend']  = df['appointment_dow'].isin([5, 6]).astype(int)
    df['is_morning']  = (df['appointment_hour'] < 12).astype(int)

    # Zaman Dilimi Parçaları
    df['appointment_datetime_month'] = df['appointment_datetime'].dt.month
    df['booking_datetime_hour']      = df['booking_datetime'].dt.hour
    df['dow_hour']                   = df['appointment_dow'] * df['appointment_hour']

    # SMS Etkileşim Metrikleri
    df['sms_lead_hours']    = df['sms_lead_hours'].fillna(0)
    df['no_sms']            = (df['sms_lead_hours'] == 0).astype(int)
    df['sms_very_early']    = (df['sms_lead_hours'] > 48).astype(int)
    df['sms_to_appt_hours'] = df['lead_time_hours'] - df['sms_lead_hours']

    # Birleşik Anahtar (Interaction Key)
    df['spec_type_key'] = df['specialty'].astype(str) + '_' + df['appointment_type'].astype(str)

    # --- Target Encoding İşlemleri ---
    global_mean = ref['label_noshow'].mean()

    df['specialty_noshow_rate'] = df['specialty'].map(
        ref.groupby('specialty')['label_noshow'].mean()).fillna(global_mean)

    df['spec_type_noshow'] = df['spec_type_key'].map(
        ref.groupby('spec_type_key')['label_noshow'].mean()).fillna(global_mean)

    for col in ['appointment_dow', 'appointment_hour', 'appointment_datetime_month']:
        df[f'{col}_noshow_rate'] = df[col].map(
            ref.groupby(col)['label_noshow'].mean()).fillna(global_mean)

    # Hasta Randevu Geçmişi Skoru
    df['patient_appointment_count'] = df['patient_id'].map(
        ref.groupby('patient_id')['label_noshow'].count()).fillna(1)

    return df

# Özelliklerin uygulanması
train = add_features(train, ref=train_raw)
test  = add_features(test,  ref=train_raw)
print("Özellik mühendisliği adımları tamamlandı.")

# %% [markdown]
# ## 4. One-Hot Encoding ve Veri Bölümleme
# Kategorik kolonlar kukla (dummy) değişkenlere çevriliyor ve veri seti Train/Validation/Test olarak ayrılıyor. Sınıf dengesizliğini çözmek için pozitif sınıf ağırlığı (`weight`) hesaplanıyor.

# %%
train = pd.get_dummies(train, columns=["booking_channel", "appointment_type"])
test  = pd.get_dummies(test,  columns=["booking_channel", "appointment_type"])

drop_cols = ['appointment_id', 'patient_id', 'appointment_datetime',
             'booking_datetime', 'specialty', 'spec_type_key']

y = train["label_noshow"]
X = train.drop(columns=[c for c in drop_cols + ['label_noshow'] if c in train.columns])
test_X = test.drop(columns=[c for c in drop_cols if c in test.columns])
test_X = test_X.reindex(columns=X.columns, fill_value=0)

# Stratified split ile sınıf oranlarının korunması
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=42, stratify=y)
weight = train_y.value_counts()[0] / train_y.value_counts()[1]

print(f"Train Şekli: {train_X.shape}, Validation Şekli: {val_X.shape}, Test Şekli: {test_X.shape}")

# %% [markdown]
# ## 5. Model Parametrelerinin Tanımlanması
# Sektör standardı en güçlü 3 gradyan artırma (gradient boosting) algoritması için parametre havuzu kurgulanıyor.

# %%
xgb_params = dict(
    n_estimators=314, learning_rate=0.06296, scale_pos_weight=weight,
    max_depth=3, subsample=0.674, colsample_bytree=0.611,
    min_child_weight=7, n_jobs=-1, eval_metric='aucpr'
)
lgbm_params = dict(
    n_estimators=300, learning_rate=0.05, max_depth=5,
    class_weight='balanced', n_jobs=-1
)
cat_params = dict(
    iterations=500, learning_rate=0.05, depth=6,
    auto_class_weights='Balanced', verbose=False
)

SEEDS = [42, 123, 456, 789, 2024]

# %% [markdown]
# ## 6. Seed Averaging ve Çoklu Model Eğitimi
# Modellerin kararlılığını artırmak için 5 farklı rastgele seed değeri üzerinden varyans azaltma eğitimi gerçekleştiriliyor.

# %%
print("Seed averaging ile modeller eğitiliyor...")

all_val_xgb,  all_val_lgbm,  all_val_cat  = [], [], []
all_test_xgb, all_test_lgbm, all_test_cat = [], [], []

for seed in SEEDS:
    print(f"  Eğitiliyor: Seed {seed}...", end=" ", flush=True)

    # XGBoost
    m_xgb = XGBClassifier(**xgb_params, random_state=seed, early_stopping_rounds=50)
    m_xgb.fit(train_X, train_y, eval_set=[(val_X, val_y)], verbose=False)
    best_iter = m_xgb.best_iteration

    m_xgb_full = XGBClassifier(**{**xgb_params, 'n_estimators': best_iter}, random_state=seed)
    m_xgb_full.fit(X, y)

    all_val_xgb.append(m_xgb.predict_proba(val_X)[:, 1])
    all_test_xgb.append(m_xgb_full.predict_proba(test_X)[:, 1])

    # LightGBM
    m_lgbm = LGBMClassifier(**lgbm_params, random_state=seed)
    m_lgbm.fit(train_X, train_y)

    m_lgbm_full = LGBMClassifier(**lgbm_params, random_state=seed)
    m_lgbm_full.fit(X, y)

    all_val_lgbm.append(m_lgbm.predict_proba(val_X)[:, 1])
    all_test_lgbm.append(m_lgbm_full.predict_proba(test_X)[:, 1])

    # CatBoost
    m_cat = CatBoostClassifier(**cat_params, random_seed=seed)
    m_cat.fit(train_X, train_y)

    m_cat_full = CatBoostClassifier(**cat_params, random_seed=seed)
    m_cat_full.fit(X, y)

    all_val_cat.append(m_cat.predict_proba(val_X)[:, 1])
    all_test_cat.append(m_cat_full.predict_proba(test_X)[:, 1])

    print("✓")

# Ortalamaların hesaplanması
val_xgb  = np.mean(all_val_xgb,  axis=0)
val_lgbm = np.mean(all_val_lgbm, axis=0)
val_cat  = np.mean(all_val_cat,  axis=0)

test_xgb  = np.mean(all_test_xgb,  axis=0)
test_lgbm = np.mean(all_test_lgbm, axis=0)
test_cat  = np.mean(all_test_cat,  axis=0)

# %% [markdown]
# ## 7. Ensemble Optimizasyonu (Ağırlık Arama)
# `Average Precision Score` metriğini maksimize edecek en ideal model birleşim oranları doğrusal arama (grid search) ile belirleniyor.

# %%
print("Ensemble ağırlıkları optimize ediliyor...")

best_score   = 0
best_weights = (0, 0, 0)

for w_xgb in np.linspace(0, 1, 21):
    for w_lgbm in np.linspace(0, 1, 21):
        w_cat = 1.0 - w_xgb - w_lgbm
        if w_cat >= -0.001:
            blended = w_xgb * val_xgb + w_lgbm * val_lgbm + w_cat * val_cat
            score   = average_precision_score(val_y, blended)
            if score > best_score:
                best_score   = score
                best_weights = (w_xgb, w_lgbm, w_cat)

print(f"\nEn İyi Validation PR-AUC Skoru: {best_score:.4f}")
print(f"Optimal Ağırlıklar -> XGB: {best_weights[0]:.2f} | LGBM: {best_weights[1]:.2f} | CAT: {best_weights[2]:.2f}")

# %% [markdown]
# ## 8. Birinci Çıktının Hazırlanması
# İlk baz model ensembling tahmini diske yazılıyor.

# %%
final_test_preds = (best_weights[0] * test_xgb +
                    best_weights[1] * test_lgbm +
                    best_weights[2] * test_cat)

pd.DataFrame({
    'appointment_id': test['appointment_id'],
    'label_noshow': final_test_preds
}).to_csv('submit_1_seed_avg.csv', index=False)
print("submit_1_seed_avg.csv başarıyla kaydedildi.")

# %% [markdown]
# ## 9. Pseudo Labeling (Yarı Gözetimli Öğrenme) ve İleri Seviye Retrain
# Modelin test verisi üzerindeki yüksek güvenli tahminleri (%85 üstü ve %10 altı olasılıklar) sahte etiket olarak eğitim setine dahil edilerek veri seti genişletiliyor.

# %%
confident_mask = (final_test_preds > 0.85) | (final_test_preds < 0.10)
pseudo_X = test_X[confident_mask].copy()
pseudo_y = pd.Series(
    (final_test_preds[confident_mask] > 0.5).astype(int),
    index=pseudo_X.index
)
print(f"Eğitime dahil edilen Pseudo Label sayısı: {confident_mask.sum()} / {len(test_X)}")

# Veri setlerinin birleştirilmesi
X_aug = pd.concat([X, pseudo_X], ignore_index=True)
y_aug = pd.concat([y.reset_index(drop=True), pseudo_y.reset_index(drop=True)], ignore_index=True)

print("Genişletilmiş veri setiyle modeller yeniden eğitiliyor...")
all_test_xgb_pl, all_test_lgbm_pl, all_test_cat_pl = [], [], []

for seed in SEEDS:
    print(f"  Eğitiliyor (Pseudo): Seed {seed}...", end=" ", flush=True)

    m_xgb_pl = XGBClassifier(**{**xgb_params, 'n_estimators': 314}, random_state=seed)
    m_xgb_pl.fit(X_aug, y_aug)
    all_test_xgb_pl.append(m_xgb_pl.predict_proba(test_X)[:, 1])

    m_lgbm_pl = LGBMClassifier(**lgbm_params, random_state=seed)
    m_lgbm_pl.fit(X_aug, y_aug)
    all_test_lgbm_pl.append(m_lgbm_pl.predict_proba(test_X)[:, 1])

    m_cat_pl = CatBoostClassifier(**cat_params, random_seed=seed)
    m_cat_pl.fit(X_aug, y_aug)
    all_test_cat_pl.append(m_cat_pl.predict_proba(test_X)[:, 1])

    print("✓")

test_xgb_pl  = np.mean(all_test_xgb_pl,  axis=0)
test_lgbm_pl = np.mean(all_test_lgbm_pl, axis=0)
test_cat_pl  = np.mean(all_test_cat_pl,  axis=0)

final_test_preds_pl = (best_weights[0] * test_xgb_pl +
                       best_weights[1] * test_lgbm_pl +
                       best_weights[2] * test_cat_pl)

# Hibrit Karışım (%50 Baz Model + %50 Pseudo Model)
final_test_preds_combined = 0.5 * final_test_preds + 0.5 * final_test_preds_pl

# Çıktıların kaydedilmesi
pd.DataFrame({'appointment_id': test['appointment_id'], 'label_noshow': final_test_preds_pl}).to_csv('submit_2_pseudo.csv', index=False)
pd.DataFrame({'appointment_id': test['appointment_id'], 'label_noshow': final_test_preds_combined}).to_csv('submit_3_combined.csv', index=False)

print("\nBütün sonuç dosyaları (submit_1, submit_2, submit_3) başarıyla üretildi.")